# 10e Which image features help which brain regions? (NSD)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a, 10b and 10d.

Development analysis, not a prespecified test. The 8 NSD people were used once in P-7, so nothing here is a claim. For AlexNet, DINOv2 and SigLIP 2:
borrowing, the low-rank map (K = 200), own ridge, the combination on the same features and the mixed combination
(map on the feature set, own ridge on AlexNet), at k = 250 and 500; encoding r over all common vertices and per named
region, and 200-way decoding.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10e-featureregions', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "draws": [13, 14, 15], "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "wall_hours": 11.0, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "features": ["alexnet", "dinov2", "siglip2"], "ks": [250, 500]}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from scipy import stats
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in FIXED["features"]:
    f = np.load(next(p for p in IN.rglob(f"*_{name}_kpca1000.npz") if p.name.startswith(("10b_", "10d_"))))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

# region names from each person's mapping files (label number -> name), checked to agree across people
ROI_NAMES = {}
for cls in FIXED["roi_classes"]:
    maps_ = [np.load(next((conv / f"{s}_roi").rglob(f"mapping_{cls}.npy")), allow_pickle=True).item() for s in SUBJECTS]
    if any(m != maps_[0] for m in maps_[1:]):
        raise ValueError(f"{cls}: label names differ between people")
    ROI_NAMES[cls] = {int(k): str(v) for k, v in maps_[0].items()}
print("ROI_MAPPING_JSON " + json.dumps(ROI_NAMES))

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in FIXED["features"]:
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        keep = np.setdiff1d(np.arange(k), held)
        pred = fit_predict(keep, held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

rows, roi_rows = [], []
stopped = None
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    mc = meas - meas.mean(0)
    def r_vertex(pred):
        pc = pred - pred.mean(0)
        return (pc * mc).sum(0) / np.sqrt((pc ** 2).sum(0) * (mc ** 2).sum(0) + 1e-30)
    def record(feature, method, k, d, pred):
        rv = r_vertex(pred)
        rows.append({"subject": s, "feature": feature, "method": method, "k": k, "draw": d, "enc_r": float(rv.mean()),
                     "top1": float(np.mean([nc.identify_top1(pred[g], meas[g]) for g in GALS]))})
        for cls in FIXED["roi_classes"]:
            lab = rois[(s, cls)]
            for v in np.unique(lab[lab > 0]):
                sel = lab == v
                roi_rows.append({"subject": s, "feature": feature, "method": method, "k": k, "draw": d, "roi_class": cls,
                                 "roi": ROI_NAMES[cls].get(int(v), str(v)), "n": int(sel.sum()), "r": float(rv[sel].mean())})
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    X = {f: (feats(f, ids[s][pool_rows[s]]), feats(f, test_ids)) for f in FIXED["features"]}
    def donor_mean(f, x):
        acc = np.zeros((len(x), V), np.float32)
        for d in donors:
            acc += ols_pred(BETA[(f, d)], x).astype(np.float32)
        return acc / len(donors)
    avg = {f: (donor_mean(f, X[f][0]), donor_mean(f, X[f][1]).astype(np.float64)) for f in FIXED["features"]}
    basis = {f: lm.prediction_basis(avg[f][0], FIXED["K"]) for f in FIXED["features"]}
    for f in FIXED["features"]:
        record(f, "avg", 0, "-", avg[f][1])
        record(f, "own_full", "all", "-", ols_pred(ols_coef(X[f][0], y_pool), X[f][1]))
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks"]:
            idx = order[:k]
            y_k = y_pool[idx].astype(np.float64)
            parts = {}
            for f in FIXED["features"]:
                a_k, a_t = avg[f][0][idx].astype(np.float64), avg[f][1]
                mean, B = basis[f]
                m_test = lm.apply_lowrank_map(lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d), a_t, mean, B)
                m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
                xk = X[f][0][idx]
                own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(xk, y_k)
                parts[("map", f)] = (m_oof, m_test)
                parts[("own", f)] = (oof(lambda tr, he: own.refit(xk[tr], y_k[tr]).predict(xk[he]), k, d), own.predict(X[f][1]))
                record(f, "lowrank_map", k, d, m_test)
                record(f, "own_ridge", k, d, parts[("own", f)][1])
            for f in FIXED["features"]:
                for name, chosen in ((f"stack_same", [("map", f), ("own", f)]), (f"stack_mixed", [("map", f), ("own", "alexnet")])):
                    if name == "stack_mixed" and f == "alexnet":
                        continue
                    w = nc.stack_fit([parts[c][0] for c in chosen], y_k)
                    record(f, name, k, d, nc.stack_apply(w, [parts[c][0] for c in chosen], [parts[c][1] for c in chosen], y_k.mean(0)))
        if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
            stopped = f"wall clock after {s} draw {d}"
            break
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "10e_scores.csv", index=False)
    pd.DataFrame(roi_rows).to_csv(C.METRICS_DIR / "10e_roi.csv", index=False)
    print(s, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")
    if stopped:
        print("STOPPED:", stopped)
        break

In [ ]:
frame, roi = pd.DataFrame(rows), pd.DataFrame(roi_rows)
def per_subject(df, metric, **sel):
    f = df
    for key, val in sel.items():
        f = f[f[key].astype(str) == str(val)]
    return f.groupby("subject")[metric].mean().reindex(SUBJECTS).to_numpy()
def ci(x):
    x = x[~np.isnan(x)]
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x)), "positive": int((x > 0).sum())}
summary = {"post_hoc": "development: NSD people already used in P-7", "roi_mapping": ROI_NAMES, "by_feature": {}, "roi": {}}
for f in FIXED["features"]:
    block = {"avg": {m: float(np.mean(per_subject(frame, m, feature=f, method="avg"))) for m in ("enc_r", "top1")},
             "own_full": {m: float(np.mean(per_subject(frame, m, feature=f, method="own_full"))) for m in ("enc_r", "top1")}}
    for k in FIXED["ks"]:
        for method in ("lowrank_map", "own_ridge", "stack_same", "stack_mixed"):
            x = per_subject(frame, "enc_r", feature=f, method=method, k=k)
            if np.isnan(x).all():
                continue
            block[f"{method}_k{k}"] = {"enc_r": float(np.nanmean(x)), "top1": float(np.nanmean(per_subject(frame, "top1", feature=f, method=method, k=k)))}
    summary["by_feature"][f] = block
for f in FIXED["features"][1:]:
    summary[f"{f}_minus_alexnet_borrowing_enc_r"] = ci(per_subject(frame, "enc_r", feature=f, method="avg") - per_subject(frame, "enc_r", feature="alexnet", method="avg"))
    summary[f"{f}_minus_alexnet_borrowing_top1"] = ci(per_subject(frame, "top1", feature=f, method="avg") - per_subject(frame, "top1", feature="alexnet", method="avg"))
# per region, k = 250 and 500 pooled: borrowing r per feature, and the map's gain over borrowing per feature
for (cls, name), g in roi.groupby(["roi_class", "roi"]):
    entry = {"n_vertices_median": float(g.n.median())}
    for f in FIXED["features"]:
        a = per_subject(g, "r", feature=f, method="avg")
        mp = np.array([g[(g.feature == f) & (g.method == "lowrank_map") & (g.subject == s)].r.mean() for s in SUBJECTS])
        entry[f] = {"avg_r": float(np.nanmean(a)), "map_gain": float(np.nanmean(mp - a)), "people": int(np.sum(~np.isnan(a)))}
    entry["dinov2_minus_alexnet_avg_r"] = entry["dinov2"]["avg_r"] - entry["alexnet"]["avg_r"]
    summary["roi"][f"{cls}|{name}"] = entry
summary["stopped"] = stopped
(C.METRICS_DIR / "10e_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("10E_SUMMARY_JSON " + json.dumps(summary, default=str))